# Gaming and Mental Health - Causal Inference Project

**By Faatimah Majothi**

## Introduction

Digital gaming has existed as far back as the 1970's and has seen a steep incline in the last decade coinciding with technological development. Concerns surrounding the impacts of digital gaming on health, both mental and physical, have existed for just as long, with a growing focus on the mental impacts in recent years. As digital gaming becomes more accessible and therefore widespread, the question of whether it directly causes or amplifies mental health issues is more crucial than ever.


In this project we will use the principles of a Causal Inference workflow to investigate the relationship between digital gaming and mental health. We aim to complete a clean data analysis project that showcases an understanding of causal inference analysis along with proficiency in the use of python libraries and machine learning models. 


As a reminder, a key point in any analysis of a relationship between two variables is that **correlation does not always indicate causation**.

**Project Scope:**
- A: Define the problem and variables:
  -	Create a detailed DAG (directional acyclical graph) using knowledge and assumptions about the data generating process (DGP).
  -	Identify the treatment, outcome, and confounders from the DAG and check against assumtions of casual inference.
- B: Import, explore and preprocess the data.
- C: Estimate treatment effects (ATE, ATT) using the chosen machine learning model(s).
- D: Validate the model results and check refuters.
- Conclusion: Interpret and communicate the results.


### **Section A:** Causal DAG and Assumptions

In [ ]:
# Setup

# Import Libraries
import pandas as pd
import numpy as np

# Inference
import dowhy
from dowhy import CausalModel

# Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,\
confusion_matrix, classification_report, ConfusionMatrixDisplay

# Set visualisation theme
from matplotlib import pyplot as plt
import seaborn as sns
plt.rcParams['figure.facecolor'] = '#F7F0DE'
plt.rcParams['axes.facecolor'] = '#FFFFFF'

# Data
import gdown
import zipfile

# Download zip file
url = 'https://drive.google.com/uc?export=download&id=1k43IO8kMviO0VFuVWAS9Km7ZATB6FEOi'
gdown.download(url, 'data.zip', quiet=False)

# Extract zip file
with zipfile.ZipFile('data.zip', 'r') as zip_ref:
    zip_ref.extractall('data_folder')

inf_data = pd.read_csv('data_folder/gaming_mental_health_10M_40features.csv')
inf_data.head(3)


Downloading...
From (original): https://drive.google.com/uc?id=1k43IO8kMviO0VFuVWAS9Km7ZATB6FEOi
From (redirected): https://drive.google.com/uc?id=1k43IO8kMviO0VFuVWAS9Km7ZATB6FEOi&confirm=t&uuid=066c2ec8-3328-4d2f-9cb2-17c940918b58
To: c:\Users\bintm\Desktop\Projects\inf-spec-project\data.zip
100%|██████████| 67.4M/67.4M [02:48<00:00, 400kB/s]


,age,gender,income,daily_gaming_hours,weekly_sessions,years_gaming,sleep_hours,caffeine_intake,exercise_hours,stress_level,...,parental_supervision,loneliness_score,aggression_score,happiness_score,bmi,screen_time_total,eye_strain_score,back_pain_score,competitive_rank,internet_quality
0,51,Female,8615,3.68,22,17,5.26,1.00,0.18,3,...,0,2.87,3.19,5.20,19.69,4.71,5.71,4.81,80,10
1,41,Female,39453,5.70,34,16,9.20,0.70,1.44,8,...,7,4.17,7.73,5.40,26.37,6.62,6.77,3.99,57,2
2,27,Male,40466,1.58,8,22,7.39,2.24,3.15,3,...,9,9.38,2.85,5.17,25.15,9.30,2.16,4.75,59,10


**Identify variables:**


- X: Digital gaming habits
- Y: Mental health (stress/depression/etc.)
- Z: Confounders:
    - Income
    - Sleep
    - Caffiene/Drugs/Alcohol
    - Age
    - Activity
    - Health
    - Social activity
    - Work/Study

###### **There are many other possible confounders/variables that impact gaming habits and mental health, but these are some key points of interest.**

**Checking assumptions of causal inference:**


- **Exchangeability:** We have listed some of the confounders that affect our target X and Y variables. We will include them in the DAG and adjust for them in the analysis where possible.
- **Overlap:** We will check that for each value or range of confounders, there is a spread of Y values. We will do this by plotting the confounders against our Y variable on a scatterplot.
- **SUTVA (Stable Unit Treatment Value Assumption):** We will assume that each individual's outcome is only affected by their own treatment status, not by the treatment status of others. We will also assume that the treatment is applied in the same way for all individuals in the study.

**Note: The dataset used in this project was sourced from [Kaggle](https://www.kaggle.com/datasets/sharmajicoder/gaming-and-mental-health) and contains synthetic data - this is a limitation of the project.**

In [ ]:
# Data features preview (present confounder data)
print(inf_data.columns)

# Digraph syntax
causal_graph = """
digraph{
    age -> income;
    study_work -> gaming_hrs;
    study_work -> income;
    sleep -> m_health;
    m_health -> sleep_hrs;
    m_health -> socialising;
    study_work -> socialising;
    online_friends -> gaming_hrs;
    age -> restrictions;
    restrictions -> gaming_hrs;
    activity -> p_health;
    study_work -> activity;
    socialising -> gaming_hrs;
    p_health -> m_health;
    gaming_hrs -> m_health;
}
"""

# Variables DataFrame syntax
variables_df = pd.DataFrame(['age', 'income', 'study_work', 'gaming_hrs', 'sleep', 'm_health', 'sleep_hrs', 'socialising', 'online_friends', 'restrictions', 'activity', 'p_health']
)

Index(['age', 'gender', 'income', 'daily_gaming_hours', 'weekly_sessions',
       'years_gaming', 'sleep_hours', 'caffeine_intake', 'exercise_hours',
       'stress_level', 'anxiety_score', 'depression_score',
       'social_interaction_score', 'relationship_satisfaction',
       'academic_performance', 'work_productivity', 'addiction_level',
       'multiplayer_ratio', 'toxic_exposure', 'violent_games_ratio',
       'mobile_gaming_ratio', 'night_gaming_ratio', 'weekend_gaming_hours',
       'friends_gaming_count', 'online_friends', 'streaming_hours',
       'esports_interest', 'headset_usage', 'microtransactions_spending',
       'parental_supervision', 'loneliness_score', 'aggression_score',
       'happiness_score', 'bmi', 'screen_time_total', 'eye_strain_score',
       'back_pain_score', 'competitive_rank', 'internet_quality'],
      dtype='str')


In [ ]:
# DAG visualisation

dag_model = CausalModel(
    data=variables_df,
    graph=causal_graph,
    treatment='gaming',
    outcome='m_health'
)
dag_model.view_model()